In [1]:
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from google.protobuf.json_format import MessageToDict

# Loading Historical Data

Loads bus delays historical .csv data from the TTC website. API code provided by TTC.

In [2]:
bus_delays = {}

# To hit our API, you'll be making requests to:
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"

url = base_url + "/api/3/action/package_show"
params = {"id": "ttc-bus-delay-data"}
package = requests.get(url, params=params).json()

for idx, resource in enumerate(package["result"]["resources"]):

    if not resource["datastore_active"]:
        url = base_url + "/api/3/action/resource_show?id=" + resource["id"]
        resource_metadata = requests.get(url).json()

        url = resource_metadata["result"]["url"]
        name = resource_metadata["result"]["name"].replace(".csv", "").replace(".xlsx", "")
        
        # Gets bus delays
        if url.endswith(".csv"):
            bus_delays[name] = pd.read_csv(url)
        elif url.endswith(".xlsx"):
            bus_delays[name] = pd.read_excel(url)




## Loading GTFS system data

Loads GTFS data from TTC website (info about routes, stops, polygons, etc.)

In [3]:
# Reads in TTC information for GTFS (Stop times, routes, trips, etc.) from Toronto Open Data and outputs all tables as dictionary of dataframes.
import zipfile
import io

gtfs_merged = {}

url = base_url + "/api/3/action/package_show"
params = {"id": "merged-gtfs-ttc-routes-and-schedules"}
package = requests.get(url, params=params).json()

# To get resource data:

for idx, resource in enumerate(package["result"]["resources"]):


    if not resource["datastore_active"]:
        url = base_url + "/api/3/action/resource_show?id=" + resource["id"]
        resource_metadata = requests.get(url).json()
        url = resource_metadata["result"]["url"]

        # Reads each file in the zip and stores it in a dictionary of dataframes.
        with zipfile.ZipFile(io.BytesIO(requests.get(url).content)) as z:
            for file in z.namelist():
                    with z.open(file) as f:
                        gtfs_merged[file] = pd.read_csv(f)


C:\Users\nikit\AppData\Local\Temp\ipykernel_14456\570640204.py:25: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  gtfs_merged[file] = pd.read_csv(f)


The following keys are the files that correspnd with the GTFS feed

In [4]:
gtfs_merged.keys()

dict_keys(['feed_info.txt', 'levels.txt', 'pathways.txt', 'routes.txt', 'shapes.txt', 'stop_times.txt', 'stops.txt', 'trips.txt', 'agency.txt', 'calendar.txt', 'calendar_dates.txt'])

Merge bus delay data frames and clean up column names

In [5]:
bus_codes = bus_delays["Code Descriptions"]
# Cleans up and symbols
bus_codes["DESCRIPTION"] = bus_codes["DESCRIPTION"].str.replace("â\x80\x93", "&")

# Combine all bus delay dataframes into one dataframe, ignoring the index and keeping only the dataframes that have yearly data.
bus_delays_data= pd.concat(
    {name: df for name, df in bus_delays.items() if "data" in name.lower() and "readme" not in name},
    ignore_index=True
    )

# Convert Code to description and consolidate the code and incide columens
bus_delays_data = bus_delays_data.merge(bus_codes, how="left", left_on="Code", right_on="CODE")

#fix mismatching column names throughout the years to match 2025 and beyond data.
bus_delays_data["Date"] = bus_delays_data["Date"].fillna(bus_delays_data["Report Date"])
bus_delays_data["Min Delay"] = bus_delays_data["Min Delay"].fillna(bus_delays_data["Delay"])
bus_delays_data["Min Gap"] = bus_delays_data["Min Gap"].fillna(bus_delays_data["Gap"])
bus_delays_data["Line"] = bus_delays_data["Line"].fillna(bus_delays_data["Route"])
bus_delays_data["Station"] = bus_delays_data["Station"].fillna(bus_delays_data["Location"])
bus_delays_data["Incident"] = bus_delays_data["Incident"].fillna(bus_delays_data["DESCRIPTION"])
    
bus_delays_data["Date"] = pd.to_datetime(bus_delays_data["Date"], errors="coerce")

bus_delays_data.drop(columns=["Report Date", "Delay", "Gap", "Location", "DESCRIPTION"], inplace=True)

#reorder columns to match the order of the columns in the 2025 data. Incident added, will edit to get relavent code name.
bus_delays_data = bus_delays_data[["Date", "Line", "Time", "Day","Station", "Code","Incident", "Min Delay", "Min Gap", "Bound", "Vehicle"]]

display(bus_delays_data.head())
display(bus_delays_data.tail())

,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle
0,2014-01-01,95,00:23:00,Wednesday,York Mills station,NaN,Mechanical,10.0,20.0,NaN,1734.0
1,2014-01-01,102,00:55:00,Wednesday,Entire run for route,NaN,General Delay,33.0,66.0,NaN,8110.0
2,2014-01-01,54,01:28:00,Wednesday,lawrence and Warden,NaN,Mechanical,10.0,20.0,NaN,7478.0
3,2014-01-01,112,01:30:00,Wednesday,Kipling Station,NaN,Emergency Services,18.0,36.0,NaN,8084.0
4,2014-01-01,24,01:37:00,Wednesday,VP and Ellesmere,NaN,Investigation,10.0,20.0,NaN,7843.0


,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle
329840,2026-09-30,21 BRIMLEY,00:07,Wednesday,BRIMLEY AND STEELES,MFDV,ON DIVERSION,37.0,57.0,N,8599.0
329841,2026-09-30,149 ETOBICOKE-BLOOR,00:33,Wednesday,BLOOR AND OLD MILL TRA,EFO,OTHER,20.0,40.0,E,3361.0
329842,2026-09-30,94 WELLESLEY,00:44,Wednesday,WELLESLEY STATION,SFDP,DISORDERLY PATRON,11.0,22.0,NaN,8469.0
329843,2026-09-30,65 PARLIAMENT,01:31,Wednesday,QUEENS QUAY AND SHERBO,MFBIO,NaN,13.0,26.0,N,8513.0
329844,2026-09-30,54 LAWRENCE EAST,01:40,Wednesday,LAWRENCE AVENUE EAST A,SFL,NaN,0.0,0.0,W,3147.0


Some of the older records don't have codes or incidents related to weather before 2025, these will be handled seperately. Bus delays with weather codes are listed below:

In [6]:
weather_bus_delays=bus_delays_data[bus_delays_data["Incident"].str.contains("weather", case=False, na=False)]
weather_bus_delays.head()

,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle
223661,2025-01-10,929 DUFFERIN EXPRESS,21:47,Friday,DUFFERIN AND ROGERS,MFWEA,WEATHER RELATED,0.0,0.0,N,9146.0
223681,2025-01-10,7 BATHURST,00:52,Friday,BATHURST AND DAVENPORT,MFWEA,WEATHER RELATED,10.0,20.0,N,9021.0
223689,2025-01-11,320 YONGE,02:15,Saturday,YONGE AND WELLESLEY,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0
223694,2025-01-11,307 BATHURST,03:10,Saturday,BATHURST AND CEDARCROF,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0
223700,2025-01-11,903 KENNEDY SCARBOROUG,06:00,Saturday,CENTENNIAL COLLEGE,MFWEA,WEATHER RELATED,0.0,0.0,S,9403.0


The following two cells are plotting the bus delay metrics.

In [7]:
# sns.set_theme(style="whitegrid")
# sns.barplot(data=weather_bus_delays,
#             x=weather_bus_delays["Date"].dt.month, y="Min Delay", hue=weather_bus_delays["Date"].dt.year, estimator="sum", errorbar=None)
# plt.xlabel("Month")
# plt.ylabel("Mean Delay (Minutes)")
# plt.title("Average Bus Delay by Month and Year")
# plt.xticks(
#     ticks=range(12),
#     labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun",
#             "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
# )


In [8]:
# sns.set_theme(style="whitegrid")
# sns.countplot(data=weather_bus_delays,
#             x=weather_bus_delays["Date"].dt.month, hue=weather_bus_delays["Date"].dt.year)
# plt.xlabel("Month")
# plt.ylabel("Number of Delays")
# plt.title("Number of Bus Delays by Month and Year")
# plt.xticks(
#     ticks=range(12),
#     labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun",
#             "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
# )

# Including Geographic Data to Delays

## Problem Statement

There is an issue with the data sets that the delay station names =/= stop names. Aditionally there are multiple stops with the same stop names (i.e. if there are 4 stops at the 4 legs of an intersection). 

Current proposed methodology: 
- For each delay, use regex (or AI implementation later on) to find stops
    - Reject rejex for now due to strings being too different in some occurances.
- Take the average lat and long to get an average stop location for the station.
- Once modeling is done on a stop basis, include route data as well.

In [9]:
display(weather_bus_delays.head(1))
display(gtfs_merged["stops.txt"][gtfs_merged["stops.txt"]["stop_name"].str.upper().str.contains("(?=.*DUFFERIN)(?=.*ROGERS)", regex=True)])

,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle
223661,2025-01-10,929 DUFFERIN EXPRESS,21:47,Friday,DUFFERIN AND ROGERS,MFWEA,WEATHER RELATED,0.0,0.0,N,9146.0


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,level_id,tts_stop_name
1460,2082,2082,Dufferin St at Rogers Rd,NaN,43.685461,-79.445973,NaN,NaN,0,NaN,NaN,1,NaN,Duffrin Street at Rogers Road
1461,2083,2083,Dufferin St at Rogers Rd,NaN,43.685673,-79.446257,NaN,NaN,0,NaN,NaN,1,NaN,Duffrin Street at Rogers Road
4112,6020,6020,Rogers Rd at Dufferin St,NaN,43.685463,-79.446245,NaN,NaN,0,NaN,NaN,1,NaN,Rogers Road at Duffrin Street
4113,6021,6021,Rogers Rd at Dufferin St,NaN,43.685645,-79.445975,NaN,NaN,0,NaN,NaN,1,NaN,Rogers Road at Duffrin Street


## 1. Stop Aggregation and Merge Current Matches

This section takes all the unique station names in the delay datafram and matches it with a stop name and location provided in the GTFS stops.txt file

In [10]:
import string
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
english_stopwords = set(stopwords.words('english'))


def clean_station_stop_name(name, replace = True):
    if pd.isna(name):
        return ""

    name = name.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    name = name.lower()  # convert to lowercase
    if replace: 
        remove_words = [
        # remove common TTC words
            " station",
            " stn",
            " bus terminal",
            " terminal",
            " loop",
            " subway",
            " st",            
            " street",    
            " southbound",
            " northbound",
            " eastbound",
            " westbound",  
            " Road",
            " road",
            " rd"
        ]

        for word in remove_words:
            name = name.replace(word, "")
    
    stations = str(name).split()

    stations_filtered = " ".join([ station for station in stations if station not in english_stopwords])

    return stations_filtered

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nikit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Filter and clean stop names to remove common stopping words. Addtiotionally, get unique stations in the bus delays 

In [11]:
ttc_stops = gtfs_merged["stops.txt"]
ttc_stops = ttc_stops.filter(items=["stop_name", "stop_lat", "stop_lon"]).dropna()
ttc_stops["stop_name"] = ttc_stops["stop_name"].str.lower()
ttc_stops["stop_name"] = ttc_stops["stop_name"].apply(clean_station_stop_name)

# Group by exact stop name and average locations.Results in unique stop names.
ttc_stops = ttc_stops.groupby("stop_name").mean()

# Get unique station names, drop everything else from df. 847 of them
station_names = pd.DataFrame(weather_bus_delays["Station"].dropna().unique(), columns=["Station"])
station_names.head()

,Station
0,DUFFERIN AND ROGERS
1,BATHURST AND DAVENPORT
2,YONGE AND WELLESLEY
3,BATHURST AND CEDARCROF
4,CENTENNIAL COLLEGE


## 2. Normalize Station Names Data

This section parses out common intersection crossings and matches the stop/station name with stops.txt

In [12]:
split_station_names = station_names[station_names["Station"].str.contains(" AND ")]["Station"].str.split(" AND ")
display(split_station_names.head())
split_station_names = [ttc_stops[ttc_stops.index.str.contains(station[0].strip(), case=False, regex=False) & ttc_stops.index.str.contains(station[1].strip(), case=False, regex=False)].assign(station_idx=station_idx) for station_idx, station in split_station_names.items()]

0       [DUFFERIN, ROGERS]
1    [BATHURST, DAVENPORT]
2       [YONGE, WELLESLEY]
3    [BATHURST, CEDARCROF]
5    [AVENUE, ST CLAIR WE]
Name: Station, dtype: object

Group all station that contain the intersection streets and get the average latitude and longitude values if the stops are within resonable distance of each other (~20m)

In [13]:
# takes non empty matches and averages the lat and lon that are close together, but it seems to work well for the data. Will merge with geopandas later
split_station_names = [group for group in split_station_names if group is not None and not group.empty]

station_names_merged = [{group["station_idx"].iloc[0]: [group["stop_lat"].mean(), group["stop_lon"].mean()]} for group in split_station_names if [
    groups["stop_lat"].max()-groups["stop_lat"].min() < 0.0002 for groups in split_station_names]]

station_names_merged = pd.DataFrame({k: v for d in station_names_merged for k, v in d.items(
)}).T.rename(columns={0: "stop_lat", 1: "stop_lon"})

Fill in the matched stations to the original df of unique station names. First do an exact match then match the cleaned up station names.

In [14]:
station_names = station_names.merge(
    station_names_merged, left_index=True, right_index=True, how="left")

print("Not null:", len(
    station_names[~station_names["stop_lat"].isnull()]))
print("Null:", len(
    station_names[station_names["stop_lat"].isnull()]))

Not null: 560
Null: 311


In [15]:
# merge cleaned up station names with merged ttc stops
station_names = station_names.merge(
    ttc_stops, left_on=station_names["Station"].apply(clean_station_stop_name), 
    right_index=True, how="left", suffixes=("", "_merged")
    )
station_names= station_names.fillna({"stop_lat": station_names["stop_lat_merged"], "stop_lon": station_names["stop_lon_merged"]}).drop(columns=["stop_lat_merged", "stop_lon_merged"])
print("Not null:", len(
    station_names[~station_names["stop_lat"].isnull()]))
print("Null:", len(
    station_names[station_names["stop_lat"].isnull()]))

Not null: 614
Null: 257


## 3. LLM Implementation

There are still many unmatched stations due to: non existent streets, mismatching names,invalid naming/location etc. This section takes the rest of the unmatched stations and runs it through an LLM which cross references TTC website and general google maps to get latitude and longitude of these stations.

Load in chat gpt generated csv matching the stop names

In [16]:
gpt_station_names = pd.read_csv("data/chatgpt_station_names.csv", index_col=0)
# merge cleaned up station names with data frame of unique station names
station_names = station_names.merge(gpt_station_names, left_index=True, right_index=True, how="left", suffixes=("", "_gpt"))
station_names= station_names.fillna({"stop_lat": station_names["stop_lat_gpt"], "stop_lon": station_names["stop_lon_gpt"]}).drop(columns=["stop_lat_gpt", "stop_lon_gpt","Station_gpt"])
display(station_names)

,Station,stop_lat,stop_lon
0,DUFFERIN AND ROGERS,43.685560,-79.446112
1,BATHURST AND DAVENPORT,43.676738,-79.415920
2,YONGE AND WELLESLEY,43.664919,-79.384585
3,BATHURST AND CEDARCROF,43.783511,-79.445589
4,CENTENNIAL COLLEGE,43.786177,-79.229530
...,...,...,...
866,WEST ROAD AND COLBORNE,43.653432,-79.465379
867,SHEPPARD AND BAKERS FI,43.754319,-79.480593
868,SOUTH AND GLEN,43.706334,-79.390488
869,SOUTH DRIVE AND GLEN,43.677632,-79.375341


All remaining na lat/lon values are either parallel streets, routes, or general areas or entire routes. There are 39 of them

In [17]:
station_names[station_names["stop_lat"].isnull()]

,Station,stop_lat,stop_lon
13,ST PHILLIPS AND MARTIN,NaN,NaN
14,ST PHILLIPS - MARTIN G,NaN,NaN
63,MORNINGSIDE AND DURIE,NaN,NaN
84,ENTIRE ROUTE,NaN,NaN
123,SHEPPARD AVE WEST AND,NaN,NaN
132,DAVISVILLE AND THE BRI,NaN,NaN
228,MILLWOOD OVERLEA TO DO,NaN,NaN
251,25 DON MILLS ROUTE,NaN,NaN
252,939 FINCH EXPRESS ROUT,NaN,NaN
253,72 PAPE ROUTE,NaN,NaN


Merge the delay data to the unique station names

In [18]:
weather_bus_delays = weather_bus_delays.merge(station_names, how="left", left_on="Station", right_on="Station")
display(weather_bus_delays)

,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle,stop_lat,stop_lon
0,2025-01-10,929 DUFFERIN EXPRESS,21:47,Friday,DUFFERIN AND ROGERS,MFWEA,WEATHER RELATED,0.0,0.0,N,9146.0,43.685560,-79.446112
1,2025-01-10,7 BATHURST,00:52,Friday,BATHURST AND DAVENPORT,MFWEA,WEATHER RELATED,10.0,20.0,N,9021.0,43.676738,-79.415920
2,2025-01-11,320 YONGE,02:15,Saturday,YONGE AND WELLESLEY,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0,43.664919,-79.384585
3,2025-01-11,307 BATHURST,03:10,Saturday,BATHURST AND CEDARCROF,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0,43.783511,-79.445589
4,2025-01-11,903 KENNEDY SCARBOROUG,06:00,Saturday,CENTENNIAL COLLEGE,MFWEA,WEATHER RELATED,0.0,0.0,S,9403.0,43.786177,-79.229530
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1328,2026-09-03,101 DOWNSVIEW PARK,08:51,Thursday,SHEPPARD AND CHESSWOOD,MFWEA,WEATHER RELATED,420.0,420.0,NaN,7075.0,43.755540,-79.474377
1329,2026-09-03,82 ROSEDALE,15:05,Thursday,SOUTH AND GLEN,MFWEA,WEATHER RELATED,108.0,110.0,NaN,8904.0,43.706334,-79.390488
1330,2026-09-04,75 SHERBOURNE,06:37,Friday,SOUTH DRIVE AND GLEN,MFWEA,WEATHER RELATED,672.0,684.0,S,8465.0,43.677632,-79.375341
1331,2026-09-04,82 ROSEDALE,06:42,Friday,SOUTH DRIVE AND GLEN,MFWEA,WEATHER RELATED,675.0,692.0,NaN,8537.0,43.677632,-79.375341


Remianing null values. Assuming the remaining values indicate entire route is delayed or portion of route delayed. How do we want to deal with entire route? Will be ignored for now but need to be incorporated

In [19]:
weather_bus_delays[weather_bus_delays["stop_lat"].isnull()]

,Date,Line,Time,Day,Station,Code,Incident,Min Delay,Min Gap,Bound,Vehicle,stop_lat,stop_lon
13,2025-01-28,52 LAWRENCE WEST,18:48,Tuesday,ST PHILLIPS AND MARTIN,MFWEA,WEATHER RELATED,0.0,0.0,E,8758.0,NaN,NaN
14,2025-01-28,952 LAWRENCE WEST EXPR,18:49,Tuesday,ST PHILLIPS - MARTIN G,MFWEA,WEATHER RELATED,0.0,0.0,E,3605.0,NaN,NaN
68,2025-02-08,77 SWANSEA,21:30,Saturday,MORNINGSIDE AND DURIE,MFWEA,WEATHER RELATED,265.0,275.0,W,7217.0,NaN,NaN
95,2025-02-09,86 SCARBOROUGH,07:48,Sunday,ENTIRE ROUTE,MFWEA,WEATHER RELATED,20.0,20.0,NaN,0.0,NaN,NaN
100,2025-02-12,77 SWANSEA,22:48,Wednesday,MORNINGSIDE AND DURIE,MFWEA,WEATHER RELATED,0.0,0.0,NaN,7211.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1170,2026-01-29,924 VICTORIA PARK EXPR,15:44,Thursday,ENTIRE ROUTE,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0,NaN,NaN
1200,2026-02-13,39 FINCH EAST,16:00,Friday,ENTIRE ROUTE,MFWEA,WEATHER RELATED,0.0,0.0,NaN,0.0,NaN,NaN
1209,2026-02-18,129 MCCOWAN NORTH,11:29,Wednesday,KENNEDY AND RANCHOR,MFWEA,WEATHER RELATED,0.0,0.0,N,3243.0,NaN,NaN
1235,2026-02-18,116 MORNINGSIDE,16:20,Wednesday,39 GUILDPARK PATHWAY,MFWEA,WEATHER RELATED,0.0,0.0,W,3472.0,NaN,NaN


Make the weather bus delays a gdf and export it for future use.

In [21]:
import geopandas as gpd

weather_bus_delays_gdf = gpd.GeoDataFrame(
    weather_bus_delays,
    geometry=gpd.points_from_xy(
        weather_bus_delays["stop_lon"], weather_bus_delays["stop_lat"]),
    crs="EPSG:4326"
)

weather_bus_delays_gdf = weather_bus_delays_gdf.to_file("weather_bus_delays_gdf.geojson", driver="GeoJSON")